# P10 本番での評価

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

シャドー運転・カナリア（割り当ての偏り SRM）・分布のずれ・人が見る抜き取りは **模擬**（乱数固定の作りもの・手順の説明用）。費用と待ち時間だけは、講師の本物の実行の記録から計算します。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 1. シャドー運転（模擬）
AI の分類をお客様に出さずに記録し、人の分類と週ごとに比べます。

In [ ]:
from collections import Counter
from common import kit, stats
on = lab("run_online", "lab10_online_eval")
rows = on.simulate()                       # 🔴 模擬: 乱数固定の作りもの
tbl = []
for w in range(1, 5):
    sub = [r for r in rows if r["week"] == w]
    crit = sum((r["human"], r["ai"]) in on.CRITICAL for r in sub)
    agree = sum(r["ai"] == r["human"] for r in sub) / len(sub)
    tbl.append([f"W{w}", len(sub), kit.pct(agree), crit, f"{100 * stats.wilson(crit, len(sub))[1]:.2f}%"])
print("シャドー運転（模擬・1〜4 週）")
print(kit.table(tbl, ["週", "件数", "人との一致", "重大な食い違い", "重大の率の上限"]))
print("決まり「重大の率の上限 < 0.2%」→ 一致率が 9 割を超えていても、満たさない")

## 2. 割り当ての偏り（SRM）を先に確かめる（模擬）
流量の 10% に新しい依頼文を出したはずが、本当に 10% になっているか。

In [ ]:
import io, contextlib
can = [r for r in rows if r["week"] in (5, 6)]
nc = sum(r["arm"] == "canary" for r in can)
nt = len(can) - nc
ec, et = len(can) * 0.1, len(can) * 0.9
chi = (nc - ec) ** 2 / ec + (nt - et) ** 2 / et
p = stats.chi2_sf(chi, 1)
print(f"割り当て: カナリア {nc} / 対照 {nt}（期待 {ec:.0f} / {et:.0f}）")
print(f"χ² = {chi:.3f}、p = {p:.3f} → {'比は正常。結果を読みに進む' if p > 0.001 else '比がおかしい。結果を読まない'}")
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    on.main([])
line = [l for l in buf.getvalue().splitlines() if "再問い合わせ率" in l][0].strip()
print(line.split("、95%区間")[0])
print("   95%区間" + line.split("、95%区間")[1])

## 3. 分布のずれ ― χ² と PSI（模擬）
6 週目に新製品が出て「使い方」の問い合わせが増えたら、試験セットは本番を代表しなくなる。

In [ ]:
c1 = Counter(r["truth"] for r in rows if r["week"] <= 4)
c6 = Counter(r["truth"] for r in rows if r["week"] == 6)
stat, dof, p = stats.chi2_homogeneity(c1, c6)
tbl = [[x, kit.pct(c1[x] / sum(c1.values())), kit.pct(c6[x] / sum(c6.values()))] for x in on.LABELS]
print(kit.table(tbl, ["種類", "1〜4 週", "6 週"]))
print(f"\nχ² = {stat:.1f}（自由度 {dof}）p = {p:.2g} / PSI = {stats.psi(c1, c6):.3f}")

def acc(ws):
    sub = [r for r in rows if r["week"] in ws]
    return sum(r["ai"] == r["truth"] for r in sub) / len(sub)
print(f"AI の正答（模擬なので正解は分かる）: 1〜4 週 {kit.pct(acc(range(1, 5)))} → 6 週 {kit.pct(acc([6]))}")

## 4. 人が見る抜き取りの二本立て（模擬）
同じ 40 件/週でも、無作為と「自信の低い順」で見つかる誤りの数が違う。

In [ ]:
import random
wk = [r for r in rows if r["week"] == 7]
errs = sum(r["ai"] != r["truth"] for r in wk)
rnd = random.Random(3).sample(wk, 40)
risk = sorted(wk, key=lambda r: (r["conf"], r["ai"] not in ("返品・交換", "修理・故障", "請求・支払い")))[:40]
f_r = sum(r["ai"] != r["truth"] for r in rnd)
f_k = sum(r["ai"] != r["truth"] for r in risk)
print(f"7 週目の AI の誤り {errs}/{len(wk)} 件")
print(f"   無作為 40 件で見つかる誤り {f_r} 件 → 全体の誤り率の推定に使える（{kit.pct(f_r / 40)}）")
print(f"   自信の低い順 40 件で見つかる誤り {f_k} 件 → 誤りを見つけるのに強いが、率の推定には使えない")

## 5. 費用と待ち時間 ― 中央値と 95 パーセント点（本物）
講師の本物の実行の記録（out/）の、1 件ごとの待ち時間と費用から計算します。

In [ ]:
rows5, per = [], {}
for lab_, f, what in [("lab03_classify_extract", "runs_classify_{}.jsonl", "分類 v1 120 件"),
                      ("lab05_agent_trajectory", "episodes_{}_k3.jsonl", "発注 10 件×3 回")]:
    for m in ("sonnet-5-5", "opus-5-5"):
        rr = kit.read_jsonl(f"{lab_}/out/{f.format(m)}")
        s = [r["seconds"] for r in rr]
        c = sum(r["cost_usd"] for r in rr) / len(rr)
        per[(what, m)] = c
        rows5.append([what, m.split("-")[0], len(rr), f"{stats.percentile(s, .5):.1f}", f"{stats.percentile(s, .95):.1f}", f"{c:.4f}"])
print(kit.table(rows5, ["実験", "モデル", "件数", "p50 秒", "p95 秒", "USD/件"]))
for m in ("sonnet-5-5", "opus-5-5"):
    c = per[("分類 v1 120 件", m)]
    print(f"分類 1 件 {m}: {c:.4f} USD → 月 1,600 件で {c * 1600:.2f} USD（正答はどちらも 120/120）")

## 6. 例題 ― 割り当て 72 対 728 の偏りを手で確かめる
期待は 10% 対 90%。χ²（自由度 1）で調べます。

In [ ]:
nc, nt, N = 72, 728, 800
ec, et = N * 0.10, N * 0.90
x1, x2 = (nc - ec) ** 2 / ec, (nt - et) ** 2 / et
print(f"期待: カナリア {ec:.0f} / 対照 {et:.0f}")
print(f"(72 − 80)² / 80 = {x1:.4f}、(728 − 720)² / 720 = {x2:.4f}")
print(f"χ² = {x1 + x2:.4f} → 自由度 1 で p = {stats.chi2_sf(x1 + x2, 1):.3f}")
print("p が 0.001 を下回ったら、割り当ての仕組みを疑い、結果は読まない")
alt = (60 - 80) ** 2 / 80 + (740 - 720) ** 2 / 720
print(f"（参考: 同じ 800 件で 60 対 740 なら χ² = {alt:.2f}、p = {stats.chi2_sf(alt, 1):.4f}）")